# NB5: ChatDev-Style Self-Repair Loop

**2-minute intro script:** Multi-agent systems should not ship the first answer blindly. In ChatDev-style collaboration, a Coder produces work, QA tests it, a Reviewer rejects it with evidence, and the Coder repairs it. The danger is unbounded autonomy: if repair has no retry budget, the system can loop forever. This notebook implements bounded repair with typed patches, typed test results, and repair memory.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from typing import List, Optional

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class CodePatch(StrictModel):
    files: dict[str, str]
    rationale: str

class TestResult(StrictModel):
    passed: bool
    error_log: str | None = None
    failing_tests: List[str] = Field(default_factory=list)

class RepairMemory(StrictModel):
    original_task: str
    attempt_number: int
    last_error: str | None
    patches_tried: List[str] = Field(default_factory=list)

In [ ]:
class BoundedRepairLoop:
    """ChatDev-style self-repair with bounded retries."""

    def __init__(self, max_repairs: int = 3):
        self.max_repairs = max_repairs
        self.memory: Optional[RepairMemory] = None

    def execute_tests(self, patch: CodePatch) -> TestResult:
        # [DETERMINISTIC MOCK] Replace with a sandboxed test runner
        # in production. This isolates the repair-loop mechanics.
        if "buggy" in patch.rationale.lower():
            return TestResult(
                passed=False,
                error_log="AssertionError: expected 2+2=4, got 5",
                failing_tests=["test_addition"],
            )
        return TestResult(passed=True)

    def repair(self, task: str, initial_patch: CodePatch) -> tuple[CodePatch, str]:
        self.memory = RepairMemory(
            original_task=task,
            attempt_number=0,
            last_error=None,
            patches_tried=[],
        )
        current_patch = initial_patch

        for attempt in range(self.max_repairs + 1):
            self.memory.attempt_number = attempt
            self.memory.patches_tried.append(current_patch.rationale)

            print(f"\n=== Attempt {attempt + 1}/{self.max_repairs + 1} ===")
            print(f"Patch: {current_patch.rationale}")

            result = self.execute_tests(current_patch)
            if result.passed:
                print(f"Tests passed on attempt {attempt + 1}")
                self.memory.last_error = None
                return current_patch, f"Success after {attempt + 1} attempts"

            self.memory.last_error = result.error_log
            print(f"Tests failed: {result.error_log}")

            if attempt >= self.max_repairs:
                print(f"Max repairs ({self.max_repairs}) reached. Escalating.")
                return current_patch, f"Failed after {self.max_repairs + 1} attempts"

            print("Generating repair based on error...")
            current_patch = CodePatch(
                files={"math.py": "def add(a, b): return a + b"},
                rationale=f"Fixed: {result.failing_tests[0] if result.failing_tests else 'error'}",
            )

        return current_patch, "Unknown error"

In [ ]:
def demo_repair_loop():
    loop = BoundedRepairLoop(max_repairs=2)
    initial_patch = CodePatch(
        files={"math.py": "def add(a, b): return a + b + 1"},
        rationale="buggy implementation that adds extra 1",
    )
    final_patch, summary = loop.repair("Implement add(a, b) function", initial_patch)

    print("\n=== Final Result ===")
    print(f"Summary: {summary}")
    print(f"Final patch: {final_patch.rationale}")
    print(f"Repair memory: {loop.memory.model_dump()}")

demo_repair_loop()

## Schema-Rejection Feedback Loop

Pydantic errors are not just exceptions. In production repair loops, they become the exact feedback prompt sent back to the LLM. The agent does not simply "try again"; it receives structured evidence about what broke.

In [ ]:
def mock_llm_with_schema_feedback(
    task: str,
    validation_error: str | None,
    attempt: int,
) -> dict:
    """Simulate an LLM that reads validation feedback before retrying."""
    if attempt == 0:
        # First attempt: LLM hallucinates an extra field.
        return {
            "files": {"math.py": "def add(a, b): return a + b + 1"},
            "rationale": "Initial buggy attempt",
            "extra_commentary": "LLMs love to add extra fields.",
        }

    # Subsequent attempts: LLM receives the Pydantic error and adapts.
    if validation_error and "extra_commentary" in validation_error:
        # It fixed the schema error but kept the logic bug.
        return {
            "files": {"math.py": "def add(a, b): return a + b + 1"},
            "rationale": (
                "Removed extra fields after seeing schema error: "
                f"{validation_error[:40]}..."
            ),
        }

    # Final successful attempt after QA feedback.
    return {
        "files": {"math.py": "def add(a, b): return a + b"},
        "rationale": "Fully repaired after schema and QA feedback.",
    }

print("--- Attempt 1: LLM hallucinates an extra field ---")
raw_output_1 = mock_llm_with_schema_feedback("Implement add", None, 0)
schema_error_msg = None

try:
    CodePatch.model_validate(raw_output_1)
except ValidationError as exc:
    schema_error_msg = str(exc)
    print(f"Pydantic blocked it: {exc.errors()[0]['msg']}")
    print("Feeding this exact error string back to the LLM for Attempt 2...\n")

print("--- Attempt 2: LLM reads the Pydantic error and fixes the schema ---")
raw_output_2 = mock_llm_with_schema_feedback("Implement add", schema_error_msg, 1)
try:
    patch_2 = CodePatch.model_validate(raw_output_2)
    print("Schema accepted. The LLM used validation feedback to fix its JSON.")
    print(f"Rationale: {patch_2.rationale}\n")
except ValidationError as exc:
    print(f"Still failing schema: {exc}")

## Typed Escalation

When repair fails, production systems should not return a vague string like "failed after 3 attempts." They should emit a typed escalation artifact that a human, manager agent, or ticketing system can parse.

In [ ]:
from typing import Literal
from uuid import uuid4

class EscalationTicket(StrictModel):
    """Typed ticket for human or manager review after bounded repair fails."""

    ticket_id: str = Field(default_factory=lambda: f"esc-{uuid4().hex[:6]}")
    original_task: str
    failed_attempts: int
    last_schema_error: str | None = None
    last_qa_error: str | None = None
    reason: Literal["max_repairs_exceeded", "token_budget_exhausted"]

class BoundedRepairLoopV2:
    def __init__(self, max_repairs: int = 2):
        self.max_repairs = max_repairs

    def run_with_typed_escalation(self, task: str) -> CodePatch | EscalationTicket:
        current_error: str | None = None
        last_schema_error: str | None = None
        last_qa_error: str | None = None

        for attempt in range(self.max_repairs + 1):
            print(f"--- LLM Attempt {attempt + 1} ---")
            raw_output = mock_llm_with_schema_feedback(task, current_error, attempt)

            try:
                patch = CodePatch.model_validate(raw_output)
                print("Schema valid. Passing to QA...")

                if "a + b + 1" in patch.files["math.py"]:
                    last_qa_error = "QA Failed: add(2,2) returned 5 instead of 4."
                    current_error = last_qa_error
                    print(f"{current_error} Feeding back to LLM...\n")

                    if attempt >= self.max_repairs:
                        break
                    continue

                print("QA passed. Shipping patch.")
                return patch

            except ValidationError as exc:
                last_schema_error = str(exc)
                current_error = last_schema_error
                print(f"Schema rejected: {exc.errors()[0]['msg']}")

        print("\nRepair budget exhausted. Generating EscalationTicket...")
        return EscalationTicket(
            original_task=task,
            failed_attempts=self.max_repairs + 1,
            last_schema_error=last_schema_error,
            last_qa_error=last_qa_error,
            reason="max_repairs_exceeded",
        )

result = BoundedRepairLoopV2(max_repairs=1).run_with_typed_escalation(
    "Implement add(a,b)"
)
print(f"\n=== Final Output Type: {type(result).__name__} ===")
print(result.model_dump_json(indent=2))
assert isinstance(result, EscalationTicket)

## Token Budget Trap

Repair loops do not only fail because of `max_repairs`. They also fail because each LLM attempt burns tokens. This bridges NB5 self-repair to NB6 cost-aware routing.

In [ ]:
class TokenBudgetTracker(StrictModel):
    total_budget: int
    used: int = 0

    def can_afford_attempt(self, estimated_tokens: int) -> bool:
        return (self.used + estimated_tokens) <= self.total_budget

def demo_token_budget_exhaustion():
    budget = TokenBudgetTracker(total_budget=1000)

    for attempt in range(5):
        estimated_tokens = 400
        if not budget.can_afford_attempt(estimated_tokens):
            print(f"Attempt {attempt + 1} blocked: token budget exhausted.")
            ticket = EscalationTicket(
                original_task="Implement add",
                failed_attempts=attempt,
                reason="token_budget_exhausted",
            )
            print(ticket.model_dump_json(indent=2))
            return ticket

        budget.used += estimated_tokens
        remaining = budget.total_budget - budget.used
        print(f"Attempt {attempt + 1} approved. Budget remaining: {remaining}")

    raise AssertionError("Budget demo should exhaust before five attempts.")

budget_ticket = demo_token_budget_exhaustion()
assert budget_ticket.reason == "token_budget_exhausted"

In [ ]:
# ==========================================
# LIVE LLM EXECUTION (Optional)
# ==========================================
# The cells above run offline using deterministic mocks.
# To see a real LLM generate output constrained by this schema:
#
#   pip install openai instructor
#   export OPENAI_API_KEY="..."
#
# Keep this False for workshops unless learners have API keys.
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    import os
    import instructor
    from openai import OpenAI

    client = instructor.from_openai(OpenAI(api_key=os.environ["OPENAI_API_KEY"]))
    model_name = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

    live_result = client.chat.completions.create(
        model=model_name,
        response_model=CodePatch,
        messages=[
            {
                "role": "user",
                "content": 'Return a CodePatch for implementing add(a, b). Include files and rationale only.',
            }
        ],
    )
    print(live_result.model_dump_json(indent=2))

## 🧪 Exercises: Bounded Autonomy

**The Story:** Your Coder agent is stuck in an infinite loop. It writes bad code, the QA agent rejects it, the Coder writes the exact same bad code, and your AWS bill hits $10,000 before you can pull the plug. We need bounded autonomy.

**Your Mission:**
1. **The Safe Sandbox:** Make `execute_tests` actually execute the code in a restricted namespace (using Python's `exec` with a blank `__builtins__` dict).
2. **The Compounding Errors:** Add a second failure before success. Observe how the `RepairMemory` accumulates the error logs and passes them to the next attempt.
3. **The Circuit Breaker:** Force all attempts to fail. Confirm that the system gracefully escalates and returns an `EscalationTicket` instead of looping forever.
4. **The Quality Gate:** Add `ReviewDecision` as a typed reviewer gate *before* the repair loop starts. If the reviewer rejects the patch outright, it shouldn't even enter the repair loop.
5. **The Institutional Memory:** Store the repair memory in the shared memory pattern from NB2. This allows other agents (like a Project Manager) to query why a task failed.

### Builder Exercise: The Bounded Writer Agent

**The Analogy:** A writer/editor pair is like a junior engineer and a code reviewer. The reviewer should not just say "bad draft." It should return typed feedback the writer can use on the next attempt.

**Semantic Building Blocks:**
- `DraftSpec`: the writer's typed artifact.
- `EditorFeedback`: the reviewer's typed correction signal.
- `SharedMemory`: the institutional brain that stores target audience and last feedback.
- `EscalationTicket`: the typed stop sign when retries run out.

**Your Mission:**
1. Store `target_audience="Technical Engineers"` in shared memory before the Writer acts.
2. Make the Writer produce a `DraftSpec` that reads the audience from memory rather than hardcoding it.
3. Make the Editor reject the first draft with `missing_elements=["Code examples"]`.
4. Feed that typed feedback into the second Writer attempt.
5. Set `max_retries_allowed=2`; force all attempts to fail once and prove the loop returns an `EscalationTicket`.

**Production Check:** A retry is not "try again." It is "try again with the exact typed evidence of what failed."

**The Takeaway:** Autonomy without boundaries is just automated chaos. Bounded repair is what makes agents safe for production.